# Bibliometrix-Python ETL Pipeline — Demonstration

**Author:** Deepak Kushwaha

**Course:** Data Science — Academic Year 2025/2026

This notebook walks through the source-agnostic ETL pipeline added to `bibliometrix-python`:
1. Loading from file sources (Scopus CSV, Dimensions XLSX, PubMed TXT)
2. Live data retrieval from APIs (OpenAlex, PubMed)
3. Schema introspection and validation
4. Re-using unmodified legacy analytical functions with the standardized DataFrame

## Cell 1 — Setup

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add the project root so the www.services.etl package is importable
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')

## Cell 2 — Import the ETL Pipeline

The pipeline exposes a single public entry point: `convert2df()` (also available as `convert_to_bibliometrix_df()`).

In [ ]:
from www.services.etl import convert2df, convert_to_bibliometrix_df
from www.services.etl.constants import TARGET_COLUMNS, LIST_FIELDS, INTEGER_FIELDS, STRING_FIELDS
from www.services.etl.dispatcher import SOURCE_REGISTRY
from www.services.etl.validation import validate_standardized_df

print('Available sources:')
for source, config in SOURCE_REGISTRY.items():
    print(f"  - {source:14s} mode={config['mode']:5s} extractor={config['extractor'].__name__}")

## Cell 3 — Schema Introspection

The pipeline enforces a 24-column WoS-style contract with strong type guarantees.

In [ ]:
import pandas as pd

schema_df = pd.DataFrame([
    {'Tag': col,
     'Type': 'list[str]' if col in LIST_FIELDS else ('int' if col in INTEGER_FIELDS else 'str'),
     'Default': '[]' if col in LIST_FIELDS else ('0' if col in INTEGER_FIELDS else '""')}
    for col in TARGET_COLUMNS
])
schema_df

## Cell 4 — File-Based Source: Scopus

Load a real Scopus CSV (1,000 records) and inspect the standardized DataFrame.

In [ ]:
scopus_df = convert2df('SCOPUS', input_path=str(ROOT / 'sources/Scopus/Scopus.csv'))
print(f'Shape: {scopus_df.shape}')
print(f'Columns: {list(scopus_df.columns)}')
print(f'No NaN: {not scopus_df.isna().any().any()}')
scopus_df[['DB','UT','TI','PY','AU','TC']].head()

## Cell 5 — File-Based Source: Dimensions

In [ ]:
dim_df = convert2df('DIMENSIONS', input_path=str(ROOT / 'sources/Dimensions/Dimensions.xlsx'))
print(f'Shape: {dim_df.shape}')
dim_df[['DB','TI','PY','AU','TC']].head()

## Cell 6 — File-Based Source: PubMed

In [ ]:
pm_df = convert2df('PUBMED_FILE', input_path=str(ROOT / 'sources/PubMed/pubmed-allergicrh-set.txt'))
print(f'Shape: {pm_df.shape}')
pm_df[['DB','PMID','TI','PY','AU','TC']].head()

## Cell 7 — Live API Query: OpenAlex

No manual download needed — the pipeline calls the OpenAlex REST API with pagination,
rate-limit handling, and exponential-backoff retries.

In [ ]:
openalex_df = convert2df('OPENALEX', query='machine learning', max_records=20)
print(f'Retrieved {len(openalex_df)} records from OpenAlex')
openalex_df[['DB','UT','TI','PY','TC']].head()

## Cell 8 — Live API Query: PubMed

In [ ]:
pubmed_api_df = convert2df('PUBMED_API', query='diabetes', max_records=20)
print(f'Retrieved {len(pubmed_api_df)} records from PubMed API')
pubmed_api_df[['DB','PMID','TI','PY']].head()

## Cell 9 — Validation

The validation module programmatically verifies that every constraint is satisfied.

In [ ]:
for label, d in [('SCOPUS', scopus_df), ('DIMENSIONS', dim_df), ('PUBMED_FILE', pm_df), ('OPENALEX', openalex_df)]:
    try:
        validate_standardized_df(d)
        print(f'✅ {label}: validation passed ({len(d)} records)')
    except Exception as e:
        print(f'❌ {label}: {e}')

## Cell 10 — Re-using Unmodified Legacy Analytics

Now feed the standardized DataFrame into existing analytical functions — they work without any modifications to their signatures.

In [ ]:
from functions.get_annualproduction import get_annual_production
from functions.get_relevantauthors import get_relevant_authors
from functions.get_bradfordlaw import get_bradford_law
from functions.get_lotkalaw import get_lotka_law
from functions.get_maininformations import get_main_informations

results = []
for label, df in [('SCOPUS', scopus_df), ('DIMENSIONS', dim_df), ('PUBMED', pm_df)]:
    for fn_name, fn, args in [
        ('annual_production', get_annual_production, ()),
        ('relevant_authors', get_relevant_authors, (10,)),
        ('bradford_law', get_bradford_law, ()),
        ('lotka_law', get_lotka_law, ()),
        ('main_informations', get_main_informations, ()),
    ]:
        try:
            fn(df.copy(), *args)
            results.append({'Source': label, 'Function': fn_name, 'Status': '✅ PASS'})
        except Exception as e:
            results.append({'Source': label, 'Function': fn_name, 'Status': f'❌ {str(e)[:40]}'})

pd.DataFrame(results)

## Summary

✅ One entry point: `convert2df()`

✅ Five sources supported: Scopus, Dimensions, PubMed (file + API), OpenAlex

✅ Strong type contracts: no NaN, list[str] for multi-value fields

✅ SR calculated field populated

✅ Validation enforces all 24 mandatory columns

✅ Legacy analytical functions work with the standardized DataFrame